# LightPolySegNet — One-click Colab training (free T4)

**Before running:** `Runtime > Change runtime type > T4 GPU`.

Pipeline: mount Drive -> get code -> download data -> train (PraNet protocol) -> evaluate -> export CSV + checkpoints to Drive.
Expected total time: ~2.5–3.5 h (100 epochs, batch 8, AMP on T4).

In [ ]:
# 1) Mount Google Drive and set paths
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_ROOT = '/content/polyp_paper'                 # working copy on Colab disk (fast)
DRIVE_BACKUP = '/content/drive/MyDrive/polyp_paper'   # persistent results
os.makedirs(DRIVE_BACKUP, exist_ok=True)
print('Project:', PROJECT_ROOT, '| Drive backup:', DRIVE_BACKUP)

In [ ]:
# 2) Get the code - search common upload locations for polyp_paper.zip
import os
PROJECT_ROOT = '/content/polyp_paper'
DRIVE_BACKUP = '/content/drive/MyDrive/polyp_paper'
os.makedirs(DRIVE_BACKUP, exist_ok=True)

candidates = [
    os.path.join(DRIVE_BACKUP, 'polyp_paper.zip'),   # MyDrive/polyp_paper/  (next to dataset.zip)
    '/content/drive/MyDrive/polyp_paper.zip',        # MyDrive root
    '/content/polyp_paper.zip',                      # direct upload into Colab
]
zip_src = next((c for c in candidates if os.path.exists(c)), None)
if zip_src:
    print('code zip found:', zip_src)
    !unzip -q -o "{zip_src}" -d /content/
else:
    print('=' * 62)
    print('未找到代码包 polyp_paper.zip ！请：')
    print('  1) 下载工作区提供的 polyp_paper.zip')
    print('  2) 上传到 Drive 的 MyDrive/polyp_paper/ 文件夹（和 dataset.zip 放一起）')
    print('  3) 重新运行本单元格')
    print('=' * 62)
    raise SystemExit('CODE ZIP NOT FOUND - see message above')
%cd {PROJECT_ROOT}
!pip install -q thop gdown timm "git+https://github.com/lartpang/PySODMetrics.git"
!ls


In [ ]:
# 3) Get data: official Polyp-PVT bundle (1.11 GB, verified 2026-10-21) ->
#    relabel into the standard 1450/798 PraNet-protocol split used by ALL baselines.
import os, glob, zipfile, shutil, subprocess, time

!pip install -q -U gdown

zip_path = '/content/dataset.zip'
DRIVE_ZIP = os.path.join(DRIVE_BACKUP, 'dataset.zip')   # manual-upload fallback
BUILD = os.path.join(PROJECT_ROOT, 'scripts', 'build_pranet_split.py')

if not os.path.exists(os.path.join(PROJECT_ROOT, 'data/TrainDataset/images')):
    if os.path.exists(DRIVE_ZIP) and os.path.getsize(DRIVE_ZIP) > 500 * 1024 * 1024:
        print('Using manually uploaded zip from Drive:', DRIVE_ZIP)
        shutil.copy(DRIVE_ZIP, zip_path)
    else:
        ok = False
        for attempt in range(1, 4):
            print(f'== download attempt {attempt}/3 ==')
            try:
                subprocess.run(['gdown', '1pFxb9NbM8mj_rlSawTlcXG1OdVGAbRQC',
                                '-O', zip_path], check=True)
                sz = os.path.getsize(zip_path)
                print(f'downloaded: {sz/1e6:.1f} MB')
                if sz < 500 * 1024 * 1024:
                    print('!! file too small - Google served an error page, retrying...')
                    time.sleep(30)
                    continue
                ok = True
                break
            except Exception as e:
                print('attempt failed:', e)
                time.sleep(30)
        if not ok:
            print('=' * 62)
            print('自动下载 3 次均失败（通常是 Google 对 Colab 出口 IP 的临时配额限制）')
            print('解决办法：')
            print('  A) 浏览器打开下面链接，登录 Google 后直接下载 dataset.zip（约1.1GB），')
            print('     上传到 Drive 的 MyDrive/polyp_paper/ 文件夹，然后重跑本单元格：')
            print('     https://drive.google.com/file/d/1pFxb9NbM8mj_rlSawTlcXG1OdVGAbRQC/view')
            print('  B) 百度网盘镜像（提取码 sydz）下载后同样上传到 MyDrive/polyp_paper/：')
            print('     https://pan.baidu.com/s/1BTgT27VxvOgKpHrigwm7Bw')
            print('=' * 62)
            raise SystemExit('DATA DOWNLOAD FAILED - follow A or B above, then re-run this cell')

    os.makedirs('/content/data_raw', exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall('/content/data_raw')
    hits = [h for h in glob.glob('/content/data_raw/**/TrainDataset', recursive=True)
            if os.path.isdir(h)]
    assert hits, '解压后未找到 TrainDataset 目录，请把本单元格完整输出截图发给团队'
    src_root = os.path.dirname(hits[0])
    print('bundle root:', src_root)

    print('Reassembling into PraNet-standard split ...')
    !python "{BUILD}" --src "{src_root}" --out "{PROJECT_ROOT}/data"

base = os.path.join(PROJECT_ROOT, 'data')
for name, n in [('TrainDataset', 1450), ('TestDataset/CVC-300', 60),
                ('TestDataset/CVC-ColonDB', 380), ('TestDataset/ETIS-Larib', 196),
                ('TestDataset/Kvasir-SEG', 100), ('TestDataset/CVC-ClinicDB', 62)]:
    imgs = len(os.listdir(os.path.join(base, name, 'images')))
    msks = len(os.listdir(os.path.join(base, name, 'masks')))
    print(name + ':', imgs, 'imgs /', msks, 'masks', 'OK' if imgs == n == msks else 'MISMATCH!')


In [ ]:
# 4) Smoke test on GPU (should pass all checks in <1 min)
!python tests/smoke_test.py

In [ ]:
# 5) Train — 换实验只改下面这一行，其余全部自动推导。
EXP_NAME = 'exp4_no_boundary'          # ← 只改这一行（可选: exp5_randinit / exp3_pretrain ...）
EXP_CONFIGS = {
    'exp1_v1baseline':  'configs/ablation_no_color.yaml',
    'exp2_coloraug':    'configs/default.yaml',
    'exp3_pretrain':    'configs/exp3_pretrain.yaml',
    'exp3_seed43':      'configs/exp3_seed43.yaml',
    'exp3_seed44':      'configs/exp3_seed44.yaml',
    'exp4_no_boundary': 'configs/ablation_no_boundary.yaml',
    'exp5_randinit':    'configs/exp5_randinit.yaml',
    'fanet_baseline':   'configs/fanet_baseline.yaml',
    'exp6_no_sobel':    'configs/exp6_no_sobel.yaml',
    'exp7_no_transformer': 'configs/exp7_no_transformer.yaml',
    'exp8_combo':       'configs/exp8_combo.yaml',
    'exp9_mobilenetv2': 'configs/exp9_mobilenetv2.yaml',
    'exp10_fpn':        'configs/exp10_fpn.yaml',
    'exp4_seed43':      'configs/ablation_no_boundary_seed43.yaml',
    'exp4_seed44':      'configs/ablation_no_boundary_seed44.yaml',
    'exp6_seed43':      'configs/exp6_no_sobel_seed43.yaml',
    'exp6_seed44':      'configs/exp6_no_sobel_seed44.yaml',
    'exp7_seed43':      'configs/exp7_no_transformer_seed43.yaml',
    'exp7_seed44':      'configs/exp7_no_transformer_seed44.yaml',
}
assert EXP_NAME in EXP_CONFIGS, f'未知实验名 {EXP_NAME}！可选: {list(EXP_CONFIGS)}'
CONFIG = EXP_CONFIGS[EXP_NAME]
OUT_DIR = f'runs/{EXP_NAME}'
DRIVE_CKPT_DIR = f'{DRIVE_BACKUP}/{EXP_NAME}_ckpt'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

RESUME = f'{DRIVE_CKPT_DIR}/last.pth' if os.path.exists(f'{DRIVE_CKPT_DIR}/last.pth') else None
print('=' * 60)
print('EXP_NAME :', EXP_NAME)
print('CONFIG   :', CONFIG)
print('CKPT DIR :', DRIVE_CKPT_DIR)
print('Resume   :', RESUME if RESUME else 'None（全新开始）')
if RESUME:
    print('!! 检测到断点将自动续训；如需全新训练，请先删除 Drive 上的', DRIVE_CKPT_DIR)
print('=' * 60)
resume_arg = f'--resume "{RESUME}"' if RESUME else ''

# ~3h for 100 epochs on T4 (bs=8, AMP)
!python scripts/train.py --config {CONFIG} \
    --data-root data/TrainDataset --out-dir {OUT_DIR} \
    --epochs 100 --batch-size 8 --val-split 0.0 {resume_arg}

import glob as _glob
_ckpt_files = _glob.glob(f'{OUT_DIR}/*.pth')
assert _ckpt_files, '训练没有产出 checkpoint——请把本单元格完整输出截图发给团队！'
!cp {OUT_DIR}/*.pth {DRIVE_CKPT_DIR}/
print('Checkpoints backed up to', DRIVE_CKPT_DIR)


In [ ]:
# 6) Evaluate the CURRENT experiment only — 不跨实验兜底，防止张冠李戴。
import os, shutil
CANDIDATES = [
    f'runs/{EXP_NAME}/last.pth',
    f'{DRIVE_CKPT_DIR}/last.pth',
]
CKPT = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert CKPT, (f'未找到 {EXP_NAME} 的 checkpoint！训练可能没有成功，'
              '请检查 cell-5 的完整输出；绝不用其他实验的权重来评估。')
print('Using checkpoint:', CKPT)

!python scripts/eval.py --checkpoint "{CKPT}" --data-root data/TestDataset \
    --out results/main_results.csv --save-vis 8
!cat results/main_results.csv
shutil.copy('results/main_results.csv', f'{DRIVE_BACKUP}/main_results_{EXP_NAME}.csv')
print('>> CSV 已按实验名存到 Drive: main_results_' + EXP_NAME + '.csv')


In [ ]:
# 7) Efficiency benchmark for the CURRENT experiment (T4, batch=1, fp32).
!python scripts/benchmark_speed.py --checkpoint "{CKPT}"


In [ ]:
# 8) Backup: vis zip + checkpoints of the CURRENT experiment to Drive.
import shutil, os, glob
vis_zip = f'{DRIVE_BACKUP}/vis_{EXP_NAME}'
if os.path.isdir('results/vis'):
    shutil.make_archive(vis_zip, 'zip', 'results/vis')
    print('Vis triptychs zipped to', vis_zip + '.zip')
else:
    print('No results/vis found (run cell-6 first).')
for f in glob.glob(f'runs/{EXP_NAME}/*.pth'):
    shutil.copy(f, DRIVE_CKPT_DIR)
print('Checkpoints backed up to', DRIVE_CKPT_DIR)
!ls -la "{DRIVE_CKPT_DIR}"
print('>> 请把 Drive 上 main_results_' + EXP_NAME + '.csv 的内容发给团队')
